# Quick Earnings Call Analysis Notebook

Configurable notebook for running the earnings call analysis pipeline. Toggle enrichment steps on/off in the config cell below. Only loads models for enabled steps — skipping topics avoids loading spaCy + sentence-transformers, which is the main speedup.

## 0. Colab Setup (auth, dependencies, models)
Run this section once per session. Authenticates with GCP, installs packages, and downloads models from GCS.

In [ ]:
# Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()
print("Authenticated.")

In [ ]:
# Install packages without touching Colab's pre-installed torch
!pip install -q --no-deps transformers sentence-transformers
!pip install -q vaderSentiment spacy huggingface-hub tokenizers safetensors
!python -m spacy download en_core_web_sm -q
print("Packages installed.")

In [ ]:
# Download models from GCS (only the ones you need)
!mkdir -p models

# Always needed for classification
!gcloud storage cp -r gs://earnings-analysis-models/models/eng_type_class_v1 ./models/
!gcloud storage cp -r gs://earnings-analysis-models/models/role_class_v1 ./models/

# Only needed if ENABLE_TOPICS = True
!gcloud storage cp -r gs://earnings-analysis-models/models/all-MiniLM-L6-v2 ./models/

print("Models downloaded.")

## Configuration

In [ ]:
# --- Date range & companies ---
START_DATE = "2026-01-15"
END_DATE   = "2026-03-26"
COMPANY_FILE = "tickers.csv"   # or set to None and use COMPANIES list
COMPANIES = None               # e.g. ["AAPL", "MSFT"] — overrides COMPANY_FILE if set

# --- Enrichment toggles (True = run, False = skip) ---
ENABLE_INTERACTION_TYPE = True   # Question / Answer / Admin
ENABLE_ROLE             = True   # Analyst / Executive / Operator / Admin
ENABLE_TOPICS           = False  # spaCy pattern matching + semantic similarity
ENABLE_SENTIMENT        = False  # VADER sentiment analysis
ENABLE_SESSIONS         = True   # Q&A session grouping

SIMILARITY_THRESHOLD = 0.7      # for topic semantic matching
BATCH_SIZE = 8                   # classification batch size

## 1. Setup

In [ ]:
import os, re, time, json, warnings
from datetime import datetime

import torch
warnings.filterwarnings('ignore', message='.*incorrect regex pattern.*', category=FutureWarning)

import pandas as pd
import google.auth
from google.cloud import bigquery
from tqdm.notebook import tqdm

# BigQuery config
BQ_PROJECT   = "sri-benchmarking-databases"
BQ_DATASET   = "pressure_monitoring"
BQ_SOURCE    = f"{BQ_PROJECT}.{BQ_DATASET}.earnings_call_transcript_content"
BQ_METADATA  = f"{BQ_PROJECT}.{BQ_DATASET}.earnings_call_transcript_metadata"
BQ_CORP_REF  = f"{BQ_PROJECT}.{BQ_DATASET}.corporation_reference"

INTERACTION_ID_MAP = {"LABEL_0": "Admin", "LABEL_1": "Answer", "LABEL_2": "Question"}
ROLE_ID_MAP = {"LABEL_0": "Admin", "LABEL_1": "Analyst", "LABEL_2": "Executive", "LABEL_3": "Operator"}

enabled = [k for k, v in {"interaction_type": ENABLE_INTERACTION_TYPE, "role": ENABLE_ROLE,
           "topics": ENABLE_TOPICS, "sentiment": ENABLE_SENTIMENT, "sessions": ENABLE_SESSIONS}.items() if v]
print(f"Enrichment: {', '.join(enabled)}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
print("Setup complete.")

## 2. Load models (only what's needed)

In [ ]:
%%time
model_dir = os.path.join(os.getcwd(), "models")
interaction_classifier = None
role_classifier = None
nlp = None
matcher = None
embedder = None
anchor_embeddings = None
anchor_metadata = []
exclusions_map = {}
issue_area_map = {}
sentiment_analyzer = None

# --- Classification models ---
if ENABLE_INTERACTION_TYPE or ENABLE_ROLE:
    from transformers import pipeline as hf_pipeline

if ENABLE_INTERACTION_TYPE:
    print("Loading interaction classifier...")
    interaction_classifier = hf_pipeline("text-classification", model=os.path.join(model_dir, "eng_type_class_v1"))

if ENABLE_ROLE:
    print("Loading role classifier...")
    role_classifier = hf_pipeline("text-classification", model=os.path.join(model_dir, "role_class_v1"))

# --- Topic detection models (spaCy + sentence-transformers) ---
if ENABLE_TOPICS:
    import spacy
    from spacy.matcher import Matcher
    from sentence_transformers import SentenceTransformer, util

    print("Loading spaCy...")
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

    print("Loading sentence-transformers embedder...")
    embedder = SentenceTransformer(os.path.join(model_dir, "all-MiniLM-L6-v2"))

    print("Loading topics.json...")
    with open("topics.json", "r", encoding="utf-8") as f:
        topics_data = json.load(f).get("topics", [])

    exclusions_map = {t['label']: t.get('exclusions', []) for t in topics_data}
    issue_area_map = {t['label']: t.get('issue_area', 'Unknown') for t in topics_data}

    matcher = Matcher(nlp.vocab)
    for topic in topics_data:
        patterns = topic.get('patterns', [])
        if patterns:
            matcher.add(topic['label'], patterns)

    all_anchors_text = []
    for topic in topics_data:
        for anchor in topic.get('anchors', []):
            all_anchors_text.append(anchor)
            anchor_metadata.append((topic['label'], anchor))

    if all_anchors_text:
        print(f"Pre-computing embeddings for {len(all_anchors_text)} anchor terms...")
        anchor_embeddings = embedder.encode(all_anchors_text, convert_to_tensor=True)

# --- Sentiment model (VADER) ---
if ENABLE_SENTIMENT:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    sentiment_analyzer = SentimentIntensityAnalyzer()
    print("VADER sentiment analyzer loaded.")

print("All required models loaded.")

## 3. Load companies & query BigQuery

In [ ]:
%%time
# Load companies
if COMPANIES:
    company_symbols = [s.strip().upper() for s in COMPANIES]
else:
    ticker_df = pd.read_csv(COMPANY_FILE).dropna(subset=['symbol'])
    company_symbols = ticker_df['symbol'].tolist()
print(f"Companies: {len(company_symbols)}")

# Auth with Drive scope (required for Drive-backed BQ tables)
credentials, _ = google.auth.default(
    scopes=[
        'https://www.googleapis.com/auth/cloud-platform',
        'https://www.googleapis.com/auth/drive',
    ]
)
client = bigquery.Client(project=BQ_PROJECT, credentials=credentials)

# Build and run query
companies_str = "', '".join(company_symbols)
query = f"""
    SELECT
        t.transcript_id, t.paragraph_number, t.speaker, t.content,
        m.* EXCEPT(transcript_id),
        cr.corporation, cr.sector
    FROM `{BQ_SOURCE}` t
    JOIN `{BQ_METADATA}` m ON t.transcript_id = m.transcript_id
    LEFT JOIN `{BQ_CORP_REF}` cr ON m.symbol = cr.Ticker
    WHERE m.symbol IN ('{companies_str}')
      AND m.report_date >= '{START_DATE}'
      AND m.report_date <= '{END_DATE}'
    ORDER BY m.report_date DESC, m.symbol, t.paragraph_number
"""

print("Running BigQuery...")
df = client.query(query).result().to_dataframe()
print(f"Fetched {len(df)} rows, {df['transcript_id'].nunique()} transcripts")

## 4. Preprocess: ensure complete transcripts, rejoin fragments, strip operator intros

In [ ]:
%%time
# Ensure complete transcripts
complete_ids = []
for tid in df['transcript_id'].unique():
    paras = sorted(df.loc[df['transcript_id'] == tid, 'paragraph_number'].unique())
    expected = list(range(paras[0], paras[0] + len(paras)))
    if paras == expected:
        complete_ids.append(tid)
df = df[df['transcript_id'].isin(complete_ids)]
print(f"Complete transcripts: {len(complete_ids)}, segments: {len(df)}")

# Rejoin fragments
rejoined = []
current = df.iloc[0].to_dict()
for i in range(1, len(df)):
    nxt = df.iloc[i].to_dict()
    msg = str(current['content']).strip()
    is_fragment = not any(msg.endswith(p) for p in ['.', '?', '!', '"', '\u201c', '\u201d'])
    if nxt['transcript_id'] == current['transcript_id'] and is_fragment:
        current['content'] = msg + " " + str(nxt['content']).strip()
    else:
        rejoined.append(current)
        current = nxt
rejoined.append(current)
df = pd.DataFrame(rejoined)
print(f"After rejoin: {len(df)} segments")

# Strip operator intros
operator_intro_re = r"^(?:We'll|We will|Let's|Certainly\.?)?\s*(?:go ahead and\s+)?(?:take|move to|now go to)\s+(?:our\s+)?(?:the\s+)?(?:first|next)?\s*question\s+(?:from|is from)\s+[^.!?]+[.!?]\s+"
texts = []
cleaned = 0
for t in df['content'].astype(str):
    c = re.sub(operator_intro_re, '', t, flags=re.IGNORECASE).strip()
    if len(c) < len(t):
        cleaned += 1
    texts.append(c if c else t)
print(f"Cleaned operator intros from {cleaned} segments")

## 5. Classify interaction type + speaker role

In [ ]:
%%time
int_results = None
role_results = None

def classify_with_progress(classifier, texts, desc):
    results = []
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc=desc):
        batch = texts[i:i + BATCH_SIZE]
        results.extend(classifier(batch))
    return results

if ENABLE_INTERACTION_TYPE or ENABLE_ROLE:
    truncated = [t[:512] for t in texts]

    if ENABLE_INTERACTION_TYPE:
        int_results = classify_with_progress(interaction_classifier, truncated, "Interaction type")

    if ENABLE_ROLE:
        role_results = classify_with_progress(role_classifier, truncated, "Speaker role")

    print("Classification complete.")
else:
    print("Classification skipped.")

## 6. Topic detection + sentiment analysis

In [ ]:
%%time
enrichment_results = None

if ENABLE_TOPICS:
    from sentence_transformers import util as st_util

    print("Running topic detection...")
    query_embeddings = embedder.encode(texts, convert_to_tensor=True)
    all_cos_scores = st_util.cos_sim(query_embeddings, anchor_embeddings) if anchor_embeddings is not None else None

    results_by_text = []
    sentiment_queue = []

    for i, text in enumerate(tqdm(texts, desc="Topic detection")):
        # 1. spaCy Matcher (exact patterns)
        doc = nlp(text)
        matches = matcher(doc)
        found_topics = set()

        if matches:
            for match_id, start, end in matches:
                found_topics.add(nlp.vocab.strings[match_id])

            text_results = []
            for topic in found_topics:
                exclusions = exclusions_map.get(topic, [])
                if any(ext.lower() in text.lower() for ext in exclusions):
                    continue
                text_results.append({"topic": topic, "idx": i})
                sentiment_queue.append({"text": text, "text_pair": topic, "text_idx": i, "topic_idx": len(text_results)-1})
            results_by_text.append(text_results)
            continue

        # 2. Vector similarity fallback
        if all_cos_scores is None:
            results_by_text.append([])
            continue

        cos_scores = all_cos_scores[i]
        text_results = []
        for idx, score in enumerate(cos_scores):
            if score.item() >= SIMILARITY_THRESHOLD:
                topic = anchor_metadata[idx][0]
                exclusions = exclusions_map.get(topic, [])
                if any(ext.lower() in text.lower() for ext in exclusions):
                    continue
                text_results.append({
                    "topic": topic, "score": score.item(),
                    "idx": i, "matched_anchor": anchor_metadata[idx][1]
                })

        text_results.sort(key=lambda x: x['score'], reverse=True)
        unique = {}
        for r in text_results:
            if r['topic'] not in unique:
                unique[r['topic']] = r
        top_topics = list(unique.values())[:3]

        for r in top_topics:
            sentiment_queue.append({
                "text": text, "text_pair": r['topic'],
                "text_idx": i, "topic_idx": top_topics.index(r)
            })
        results_by_text.append(top_topics)

    # Sentiment on detected topics
    if ENABLE_SENTIMENT and sentiment_queue:
        print(f"Running VADER sentiment on {len(sentiment_queue)} topic pairs...")
        for item in tqdm(sentiment_queue, desc="Sentiment"):
            scores = sentiment_analyzer.polarity_scores(str(item["text"]))
            compound = scores['compound']
            label = 'positive' if compound >= 0.05 else ('negative' if compound <= -0.05 else 'neutral')
            target = results_by_text[item["text_idx"]][item["topic_idx"]]
            target["sentiment"] = label
            target["sentiment_score"] = abs(compound)
            target["all_scores"] = f"pos: {scores['pos']:.2f}, neu: {scores['neu']:.2f}, neg: {scores['neg']:.2f}, compound: {compound:.2f}"

    enrichment_results = results_by_text
    print(f"Topics found in {sum(1 for r in enrichment_results if r):,}/{len(texts):,} segments")

elif ENABLE_SENTIMENT:
    # Standalone sentiment (no topics)
    print("Running standalone VADER sentiment...")
    enrichment_results = []
    for text in tqdm(texts, desc="Sentiment"):
        scores = sentiment_analyzer.polarity_scores(str(text))
        compound = scores['compound']
        label = 'positive' if compound >= 0.05 else ('negative' if compound <= -0.05 else 'neutral')
        enrichment_results.append([{
            "topic": None, "sentiment": label,
            "sentiment_score": abs(compound),
            "all_scores": f"pos: {scores['pos']:.2f}, neu: {scores['neu']:.2f}, neg: {scores['neg']:.2f}, compound: {compound:.2f}",
            "similarity_score": None, "matched_anchor": None
        }])
    print("Sentiment complete.")

else:
    print("Topics & sentiment skipped.")

## 7. Assemble results

In [ ]:
%%time
session_re = None
if ENABLE_SESSIONS:
    session_re = re.compile(
        r"next question|first question|question (?:is |will be )?(?:coming )?from|"
        r"(?:we'll |we will )?(?:now )?(?:go to|take)|"
        r"move to the line of|go to the line of|from the line of",
        re.IGNORECASE
    )

QUESTION_INDICATORS = [
    "have two", "have a question", "wondering", "curious", "want to ask",
    "can you", "could you", "would you", "will you",
    "how do", "how does", "how should", "how would",
    "what", "why", "when", "where", "which",
    "talk about", "comment on", "thoughts on", "perspective on"
]

rows = []
session_id = 0
last_tid = None
prev_int, prev_role = None, None

for i, (_, row) in enumerate(tqdm(list(df.iterrows()), desc="Assembling")):
    if last_tid is not None and row['transcript_id'] != last_tid:
        session_id = 0
        prev_int, prev_role = None, None
    last_tid = row['transcript_id']

    text = texts[i]
    interaction_type = None
    role_label = None

    if ENABLE_INTERACTION_TYPE and int_results:
        interaction_type = INTERACTION_ID_MAP.get(int_results[i]['label'], int_results[i]['label'])
    if ENABLE_ROLE and role_results:
        role_label = ROLE_ID_MAP.get(role_results[i]['label'], role_results[i]['label'])

    # Post-processing (requires both interaction + role)
    if ENABLE_INTERACTION_TYPE and ENABLE_ROLE and interaction_type and role_label:
        if interaction_type == "Answer":
            if prev_int != "Question" or prev_role != "Analyst":
                interaction_type = "Admin"
        if role_label == "Analyst" and interaction_type != "Question":
            lower = text.lower()
            if any(ind in lower for ind in QUESTION_INDICATORS):
                interaction_type = "Question"

    # Session tracking
    if ENABLE_SESSIONS:
        lower = text.lower()
        is_operator = role_label == "Operator" if role_label else False
        if (is_operator and any(k in lower for k in ["question", "line of", "analyst"])) or session_re.search(lower):
            session_id += 1

    # Get enrichment data for this segment
    if enrichment_results:
        detected = enrichment_results[i]
    else:
        detected = None

    if not detected:
        detected = [{"topic": None, "sentiment": None, "sentiment_score": None,
                      "all_scores": None, "similarity_score": None, "matched_anchor": None}]

    for d in detected:
        res = {
            "transcript_id": row['transcript_id'],
            "paragraph_number": row['paragraph_number'],
            "speaker": row['speaker'],
        }
        if ENABLE_SESSIONS:
            res["qa_session_id"] = session_id
        if ENABLE_INTERACTION_TYPE:
            res["interaction_type"] = interaction_type
        if ENABLE_ROLE:
            res["role"] = role_label
        if ENABLE_TOPICS:
            res["issue_area"] = issue_area_map.get(d.get('topic'), "Unknown")
            res["issue_subtopic"] = d.get('topic')
            res["similarity_score"] = d.get('similarity_score')
            res["matched_anchor"] = d.get('matched_anchor')
        if ENABLE_SENTIMENT:
            res["sentiment_label"] = d.get('sentiment')
            res["sentiment_score"] = d.get('sentiment_score')
            res["all_scores"] = d.get('all_scores')

        res["content"] = text
        for col in row.index:
            if col not in ('transcript_id', 'paragraph_number', 'speaker', 'content'):
                res[col] = row[col]
        rows.append(res)

    prev_int, prev_role = interaction_type, role_label

results_df = pd.DataFrame(rows)
print(f"Assembled {len(results_df):,} rows")

## 8. Save & preview

In [ ]:
os.makedirs("outputs", exist_ok=True)
output_path = f"outputs/quick_analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
results_df.to_csv(output_path, index=False)

size_mb = os.path.getsize(output_path) / (1024**2)
print(f"Saved to {output_path} ({size_mb:.2f} MB)")
print(f"  Rows: {len(results_df):,}")
print(f"  Transcripts: {results_df['transcript_id'].nunique()}")

if 'role' in results_df.columns:
    print(f"\n  Role distribution:")
    print(results_df['role'].value_counts().to_string(header=False))
if 'interaction_type' in results_df.columns:
    print(f"\n  Interaction type distribution:")
    print(results_df['interaction_type'].value_counts().to_string(header=False))
if 'issue_subtopic' in results_df.columns:
    print(f"\n  Topics detected: {results_df['issue_subtopic'].notna().sum():,}")
if 'sentiment_label' in results_df.columns:
    print(f"\n  Sentiment distribution:")
    print(results_df['sentiment_label'].value_counts().to_string(header=False))

results_df.head(10)